# 🎬 Cineforge — Kaggle GPU Worker

Same role as the Colab notebook: a free Kaggle T4/P100 becomes a Cineforge render worker that pulls jobs from your shared Redis queue.

**Kaggle setup before running:**
1. Notebook → Settings → **Accelerator: GPU T4 x2** (or P100)
2. Notebook → Settings → **Internet: On** (required for Redis/Postgres/model downloads)
3. Store secrets via **Add-ons → Secrets** (`DATABASE_URL`, `CELERY_BROKER_URL`).

In [ ]:
!nvidia-smi

## 1. Configuration (pull secrets from Kaggle Secrets)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()

REPO_URL = "https://github.com/<you>/AI_Cenimatic_Project.git"
LLM_MODEL = "llama3"

os.environ["DATABASE_URL"]          = sec.get_secret("DATABASE_URL")
os.environ["CELERY_BROKER_URL"]     = sec.get_secret("CELERY_BROKER_URL")
os.environ["CELERY_RESULT_BACKEND"] = os.environ["CELERY_BROKER_URL"]  # Upstash free = single db; reuse same url
os.environ["REDIS_URL"]             = os.environ["CELERY_BROKER_URL"]  # Upstash free = single db; reuse same url
os.environ["OLLAMA_HOST"]  = "http://127.0.0.1:11434"
os.environ["COMFYUI_URL"]  = "http://127.0.0.1:8188"
os.environ["CINEFORGE_LLM_MODEL"]    = LLM_MODEL
os.environ["CINEFORGE_ANIM_BACKEND"] = "comfyui"
os.environ["CINEFORGE_VRAM_GB"]      = "15"
os.environ["STORAGE_ROOT"]           = "/kaggle/working/AI_Cenimatic_Project/storage"
print("config set")

## 2. Clone + install

In [ ]:
%cd /kaggle/working
import os, sys
if not os.path.isdir("AI_Cenimatic_Project"): os.system(f"git clone {REPO_URL} AI_Cenimatic_Project")
%cd /kaggle/working/AI_Cenimatic_Project
!pip install -q diffusers accelerate safetensors imageio-ffmpeg moviepy httpx pillow numpy redis celery sqlalchemy psycopg2-binary pydantic pydantic-settings
!pip install -q coqui-tts || echo 'voice unavailable -> silent'
!pip install -q audiocraft || echo 'music unavailable -> silent'
!pip install -q -e packages/ai_engine
paths = ["/kaggle/working/AI_Cenimatic_Project", "/kaggle/working/AI_Cenimatic_Project/apps/api", "/kaggle/working/AI_Cenimatic_Project/packages/ai_engine"]
for p in paths:
    if p not in sys.path: sys.path.insert(0, p)
os.environ["PYTHONPATH"] = ":".join(paths)
print("installed")

## 3. Ollama + ComfyUI + SDXL

In [ ]:
import subprocess, time, httpx, os
os.system("apt-get -qq install -y zstd")
os.system("curl -fsSL https://ollama.com/install.sh | sh")
subprocess.Popen(["ollama", "serve"]); time.sleep(8)
os.system(f"ollama pull {os.environ['CINEFORGE_LLM_MODEL']}")

%cd /kaggle/working/AI_Cenimatic_Project/comfyui
if not os.path.isdir("ComfyUI"): os.system("git clone https://github.com/comfyanonymous/ComfyUI.git")
os.system("pip install -q -r ComfyUI/requirements.txt")
for repo in ["ComfyUI-AnimateDiff-Evolved", "ComfyUI-VideoHelperSuite"]:
    d = f"ComfyUI/custom_nodes/{repo}"
    if not os.path.isdir(d): os.system(f"git clone https://github.com/Kosinkadink/{repo}.git {d}")
os.system("wget -nc -q -O ComfyUI/models/checkpoints/sd_xl_base_1.0.safetensors https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors")
subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--lowvram"], cwd="ComfyUI")
for _ in range(60):
    try: httpx.get(os.environ['COMFYUI_URL']+'/system_stats', timeout=2); print('ComfyUI up'); break
    except Exception: time.sleep(3)

## 4. Start the worker

In [ ]:
%cd /kaggle/working/AI_Cenimatic_Project
!python -m gpu_worker